In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy
import os

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 이미지 처리
from skimage import io
import requests
from PIL import Image
from io import BytesIO

# tensorflow 모델
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam

# Utils

In [ ]:
# Helper: URL에서 이미지 로드 & (224×224)으로 리사이즈
def load_and_preprocess(url, idx=None, target_size=(224,224)):
    try:
        response = requests.get(url, stream=True, timeout=180)
        img = Image.open(BytesIO(response.content)).resize(target_size)
        img = img.convert("RGB")
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)
        return img_array
    except Exception as e:
        print(f"[{idx}] Error fetching image: {e}")
        # 에러 인덱스 기록
        if idx is not None:
            error_indices.append(idx)
        img_array = np.zeros((*target_size, 3), dtype=np.float32)
        img_array = preprocess_input(img_array)
        return img_array

In [ ]:
# batch 단위로 feature 추출
def predict_batch(url_list, index_list, batch_size=4, model=None):

    # 1. 이미지 불러오기 및 전처리
    image_arrays = [load_and_preprocess(url, idx) for url, idx in zip(url_list, index_list)]
    batch_array = np.stack(image_arrays, axis=0)

    # 2. 배치 단위로 예측
    preds = model.predict(batch_array, batch_size=batch_size, verbose=1)
    return preds

In [ ]:
# Top 5 core Filtering
def top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자 필터링
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        data = data[~data['user_id'].isin(users_to_remove)]

        # 아이템 필터링
        item_counts = data['asin'].value_counts()
        items_to_remove = item_counts[item_counts < chk].index
        data = data[~data['asin'].isin(items_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        # 두 조건 모두 만족 시 종료
        if (len(users_to_remove) == 0) and (len(items_to_remove) == 0):
            break

        cnt += 1

    return data

# Load Dataset

In [ ]:
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v1.json')
print(df.shape)
df.head()

(67248, 10)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...


# Extract VGG16 Image Embedding

In [ ]:
# Load VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='max')
x = Flatten()(base_model.output)
predictions = Dense(4096, activation='relu')(x)
model = Model(inputs=base_model.input, outputs=predictions)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
# Apply the function to each row
batch_size = 8
all_image_urls = df['medium_image_url'].tolist()
all_indices = df.index.tolist()  # 실제 DataFrame index

error_indices = []
results = []
for i in tqdm(range(0, len(all_image_urls), batch_size)):
    batch_urls = all_image_urls[i:i+batch_size]
    batch_indices = all_indices[i:i+batch_size]
    preds = predict_batch(batch_urls, batch_indices, batch_size=len(batch_urls), model=model)
    results.extend(preds)

embedding = pd.DataFrame({'VGG16_emb': [e for e in results]})
df = pd.concat([df, embedding], axis=1)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# error_indices에 오류난 데이터 인덱스가 다 들어있음
print(f"\n불러오지 못한 이미지 인덱스: {error_indices}")
print(f"\n총 오류 개수: {len(error_indices)}")
print("\n실패한 url:")

for idx in error_indices:
    print(df.loc[idx, 'medium_image_url'])


불러오지 못한 이미지 인덱스: [866, 867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 888, 889, 890, 891, 892, 893, 894, 895, 896, 9516, 9517, 9518, 9519, 9520, 9521, 9522, 9523, 9524, 9525, 9526, 9527, 9528, 18240, 18241, 18242, 18243, 18288, 18289, 18290, 18291, 18292, 18293, 18294, 18295, 18296, 18297, 18298, 18299, 18300, 18301, 18302, 18303, 18304, 18305, 18306, 18307, 18308, 18309, 18310, 18311, 18312, 23293, 23294, 23295, 23296, 23297, 23673, 23674, 23675, 23676, 23677, 23678, 23679, 29186, 29187, 29188, 29189, 29190, 29191, 29192, 29193, 29194, 29195, 29196, 29197, 29198, 31723, 31724, 31725, 31726, 31727, 31728, 31729, 31730, 31731, 31732, 31733, 31734, 31735, 31736, 31737, 31738, 31739, 31740, 31741, 31742, 31743, 31744, 31745, 31746, 45358, 45359, 45360, 45361, 45362, 45363, 45364, 45365, 45366, 45367, 45368, 45369, 45370, 45371, 45372, 51619, 51620, 51621, 51622, 51623, 51624, 51625, 51626, 51627, 62937, 62938, 62939, 62940, 65008,

In [ ]:
len(error_indices)

255

In [ ]:
df.shape

(67248, 11)

In [ ]:
df = df.drop(index=error_indices)
top3 = top_n_filtering(df, 3)

# index 초기화
top3 = top3.reset_index(drop=True)
top3.shape

Epoch 1
 - Remaining users: 7475
 - Remaining items: 16121
 - Remaining reviews: 66814
Epoch 2
 - Remaining users: 7458
 - Remaining items: 16104
 - Remaining reviews: 66746
Epoch 3
 - Remaining users: 7455
 - Remaining items: 16102
 - Remaining reviews: 66736
Epoch 4
 - Remaining users: 7452
 - Remaining items: 16100
 - Remaining reviews: 66726
Epoch 5
 - Remaining users: 7449
 - Remaining items: 16099
 - Remaining reviews: 66718
Epoch 6
 - Remaining users: 7449
 - Remaining items: 16099
 - Remaining reviews: 66718


(66718, 11)

In [ ]:
top3.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v1.json')